# Chapter 12 Companion Notebook: Turbulent Boundary Layer

**Principles of Turbulence — Python Companion Computations**

This notebook supports the chapter on turbulent boundary layers. It provides
compact computational demonstrations of:

1. boundary-layer growth over a flat plate,
2. inner and outer scaling of mean velocity,
3. displacement thickness, momentum thickness, and shape factor,
4. skin-friction scaling and Reynolds-number effects,
5. von Kármán integral momentum balance,
6. schematic turbulent kinetic-energy budget behavior,
7. entrainment and outer-layer growth,
8. Reynolds-stress anisotropy visualization using invariant maps.

The examples use synthetic or canonical profiles so that the physical ideas remain transparent.

## 1. Imports and Plot Settings

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (7, 4.5),
    "font.size": 11,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

## 2. Boundary-Layer Growth over a Smooth Flat Plate

For a zero-pressure-gradient turbulent boundary layer, a useful empirical scaling is

\[
\delta(x) \sim 0.37 x Re_x^{-1/5},
\qquad
Re_x = \frac{U_e x}{\nu}.
\]

This captures the slow downstream growth of a high-Reynolds-number turbulent boundary layer.

In [ ]:
Ue = 20.0          # free-stream velocity [m/s]
nu = 1.5e-5       # air kinematic viscosity [m^2/s]
x = np.linspace(0.05, 5.0, 500)

Re_x = Ue * x / nu
delta_turb = 0.37 * x * Re_x**(-1/5)
delta_lam = 5.0 * x / np.sqrt(Re_x)

plt.figure()
plt.plot(x, delta_lam * 1000, label="Laminar estimate")
plt.plot(x, delta_turb * 1000, label="Turbulent estimate")
plt.xlabel(r"$x$ [m]")
plt.ylabel(r"$\delta$ [mm]")
plt.title("Boundary-Layer Growth over a Flat Plate")
plt.legend()
plt.show()

## 3. Synthetic Mean Velocity Profiles

A simple composite turbulent profile can be constructed using inner coordinates.
The objective is not to reproduce a specific DNS dataset, but to visualize the
layered structure: viscous sublayer, buffer layer, logarithmic region, and outer region.

In [ ]:
kappa = 0.41
B = 5.2

y_plus = np.logspace(0, 4, 500)

U_visc = y_plus
U_log = (1/kappa) * np.log(y_plus) + B

blend = 1 / (1 + np.exp(-(np.log(y_plus) - np.log(11)) / 0.35))
U_plus = (1 - blend) * U_visc + blend * U_log

plt.figure()
plt.semilogx(y_plus, U_plus, label="Synthetic turbulent profile")
plt.semilogx(y_plus, U_visc, "--", label=r"$U^+=y^+$")
plt.semilogx(y_plus, U_log, "--", label=r"$U^+=\kappa^{-1}\ln y^+ + B$")
plt.axvspan(1, 5, alpha=0.15, label="Viscous sublayer")
plt.axvspan(5, 30, alpha=0.10, label="Buffer layer")
plt.xlabel(r"$y^+$")
plt.ylabel(r"$U^+$")
plt.title("Inner-Scaled Mean Velocity Profile")
plt.legend()
plt.show()

## 4. Displacement Thickness, Momentum Thickness, and Shape Factor

For an external boundary layer,

\[
\delta^*=\int_0^\infty \left(1-\frac{U}{U_e}\right)\,dy,
\qquad
\theta=\int_0^\infty \frac{U}{U_e}\left(1-\frac{U}{U_e}\right)\,dy,
\qquad
H=\frac{\delta^*}{\theta}.
\]

The displacement thickness measures the mass-flow deficit, while the momentum
thickness measures the momentum deficit.

In [ ]:
def power_law_profile(eta, n=7):
    '''Approximate turbulent boundary-layer profile: U/Ue = eta^(1/n).'''
    eta = np.clip(eta, 0, 1)
    return eta**(1/n)

eta = np.linspace(0, 1, 2000)
delta = 0.05  # representative boundary-layer thickness [m]
y = eta * delta
U_Ue = power_law_profile(eta, n=7)

delta_star = np.trapz(1 - U_Ue, y)
theta = np.trapz(U_Ue * (1 - U_Ue), y)
H = delta_star / theta

print(f"delta* = {delta_star:.5f} m")
print(f"theta  = {theta:.5f} m")
print(f"H      = {H:.3f}")

plt.figure()
plt.plot(U_Ue, eta)
plt.xlabel(r"$U/U_e$")
plt.ylabel(r"$y/\delta$")
plt.title("Approximate Turbulent Boundary-Layer Profile")
plt.show()

## 5. Skin-Friction Scaling and Reynolds-Number Effects

A common turbulent flat-plate estimate is

\[
C_f(x) \sim Re_x^{-1/5}.
\]

This scaling captures the slow decrease of local wall shear stress with increasing
downstream Reynolds number.

In [ ]:
Cf_turb = 0.0592 * Re_x**(-1/5)
Cf_lam = 0.664 / np.sqrt(Re_x)

plt.figure()
plt.loglog(Re_x, Cf_lam, label=r"Laminar: $0.664 Re_x^{-1/2}$")
plt.loglog(Re_x, Cf_turb, label=r"Turbulent: $0.0592 Re_x^{-1/5}$")
plt.xlabel(r"$Re_x$")
plt.ylabel(r"$C_f$")
plt.title("Local Skin-Friction Coefficient")
plt.legend()
plt.show()

## 6. von Kármán Integral Momentum Balance

For a zero-pressure-gradient turbulent boundary layer,

\[
\frac{d\theta}{dx} = \frac{C_f}{2}.
\]

Using a skin-friction correlation, the momentum thickness can be estimated by
integrating \(C_f/2\) downstream.

In [ ]:
dtheta_dx = Cf_turb / 2

theta_x = np.zeros_like(x)
theta_x[1:] = np.cumsum(0.5 * (dtheta_dx[1:] + dtheta_dx[:-1]) * np.diff(x))

plt.figure()
plt.plot(x, theta_x * 1000)
plt.xlabel(r"$x$ [m]")
plt.ylabel(r"$\theta$ [mm]")
plt.title(r"Momentum Thickness from $d\theta/dx=C_f/2$")
plt.show()

## 7. Schematic TKE Budget Across the Boundary Layer

This plot is schematic. It illustrates typical trends:

- dissipation is important close to the wall,
- production peaks in the buffer layer,
- production and dissipation are approximately balanced in the log region,
- transport and entrainment become important in the outer layer.

In [ ]:
yp = np.logspace(0, 4, 600)
logyp = np.log10(yp)

P = np.exp(-0.5*((logyp - np.log10(15))/0.35)**2)
eps = 0.35 + 0.55*np.exp(-0.5*((logyp - np.log10(3))/0.55)**2)
transport = 0.25*np.exp(-0.5*((logyp - np.log10(20))/0.45)**2) + 0.55*(yp/yp.max())**0.25

P /= P.max()
eps /= eps.max()
transport /= transport.max()

plt.figure()
plt.semilogx(yp, P, label="Production")
plt.semilogx(yp, eps, label="Dissipation")
plt.semilogx(yp, transport, label="Transport / entrainment")
plt.axvspan(1, 5, alpha=0.12, label="Viscous")
plt.axvspan(5, 30, alpha=0.10, label="Buffer")
plt.axvspan(30, 1000, alpha=0.07, label="Log")
plt.xlabel(r"$y^+$")
plt.ylabel("Relative magnitude")
plt.title("Schematic TKE-Budget Structure")
plt.legend()
plt.show()

## 8. Entrainment and Outer-Layer Growth

For a zero-pressure-gradient boundary layer, entrainment can be related schematically to

\[
v_e \sim U_e \frac{d\delta^*}{dx}.
\]

Although \(v_e/U_e\) is small, it controls the slow ingestion of free-stream fluid
into the turbulent boundary layer.

In [ ]:
delta_star_x = 0.125 * delta_turb
d_delta_star_dx = np.gradient(delta_star_x, x)
ve = Ue * d_delta_star_dx

plt.figure()
plt.plot(x, ve / Ue)
plt.xlabel(r"$x$ [m]")
plt.ylabel(r"$v_e/U_e$")
plt.title("Schematic Entrainment Velocity Scaling")
plt.show()

print(f"Typical v_e/Ue range: {ve.min()/Ue:.4e} to {ve.max()/Ue:.4e}")

## 9. Reynolds-Stress Anisotropy and Invariant Map

The Reynolds-stress anisotropy tensor is

\[
b_{ij}=\frac{\overline{u_i'u_j'}}{2k}-\frac{1}{3}\delta_{ij}.
\]

The invariants

\[
\mathrm{II}_b=b_{ij}b_{ji},
\qquad
\mathrm{III}_b=b_{ij}b_{jk}b_{ki}
\]

give a coordinate-independent description of turbulence anisotropy.

In [ ]:
def invariants_from_b(b):
    II = np.trace(b @ b)
    III = np.trace(b @ b @ b)
    return II, III

s = np.linspace(1, 0.05, 100)
II_vals = []
III_vals = []

for a in s:
    b = np.diag([2*a/3, -a/3, -a/3])
    II, III = invariants_from_b(b)
    II_vals.append(II)
    III_vals.append(III)

plt.figure()
plt.plot(II_vals, III_vals, label="Illustrative wall-normal trajectory")
plt.scatter([0], [0], label="Isotropy")
plt.xlabel(r"$\mathrm{II}_b$")
plt.ylabel(r"$\mathrm{III}_b$")
plt.title("Illustrative Reynolds-Stress Anisotropy Trajectory")
plt.legend()
plt.show()

## 10. Suggested Exercises

1. Change \(U_e\) and observe how \(Re_x\), \(C_f\), and \(\delta(x)\) change.
2. Replace the \(1/7\)-power profile with another approximate profile and recompute
   \(\delta^*\), \(\theta\), and \(H\).
3. Modify the schematic TKE profiles and identify how the layer interpretation changes.
4. Compare laminar and turbulent drag estimates for the same plate length.
5. Use DNS or experimental boundary-layer data to replace the synthetic profiles.